In [ ]:
# Setup
import sys
import json
import time
import requests
import redis
from pathlib import Path

# Project root
current_dir = Path.cwd()
if current_dir.name == "module7" and current_dir.parent.name == "notebooks":
    project_root = current_dir.parent.parent
else:
    project_root = Path("/mnt/data/Portfolio/RAG")

sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

API = "http://localhost:8000/api/v1"

# Health check
print("\nMODULE 7 HEALTH CHECK")
print("=" * 40)

services = {
    "FastAPI" : "http://localhost:8000/api/v1/health",
    "OpenSearch": "http://localhost:9200/_cluster/health",
    "Ollama"  : "http://localhost:11434/api/version",
    "Langfuse": "http://localhost:3000/api/public/health",
}

for name, url in services.items():
    try:
        r = requests.get(url, timeout=5)
        print(f"{'✓' if r.status_code == 200 else '✗'} {name}")
    except Exception as e:
        print(f"✗ {name}: {e}")

try:
    redis.Redis(host="localhost", port=6379, socket_connect_timeout=3).ping()
    print("✓ Redis")
except Exception as e:
    print(f"✗ Redis: {e}")

## Multi-Turn Dialogue

Every `/ask` and `/stream` request now supports an optional `session_id`.

| | Stateless | With `session_id` |
|---|---|---|
| History loaded | — | ✓ last 5 turns |
| Exact-match cache | ✓ | — |
| `session_id` in response | new UUID | echoed back |

Pass the returned `session_id` in the next request to continue the conversation.

In [ ]:
# Step 1 — First request (no session_id)
print("STEP 1 — STATELESS REQUEST")
print("=" * 40)
print("No session_id sent. The API generates one and returns it.\n")

resp = requests.post(f"{API}/ask", json={
    "query"     : "What is retrieval-augmented generation?",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
}, timeout=120)

data = resp.json()
session_id = data["session_id"]

print(f"session_id : {session_id}")
print(f"chunks_used: {data['chunks_used']}")
print(f"cached     : {data.get('cached', False)}")
print(f"\nAnswer:\n{data['answer'][:500]}...")

In [ ]:
# Step 2 — Follow-up (pass session_id)
print("STEP 2 — FOLLOW-UP REQUEST")
print("=" * 40)
print(f"Reusing session_id: {session_id}")
print("The model now sees the previous Q&A as context.\n")

resp2 = requests.post(f"{API}/ask", json={
    "query"     : "Can you give a concrete example of how it works?",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    "session_id": session_id,
}, timeout=120)

data2 = resp2.json()
print(f"session_id echoed: {data2['session_id'] == session_id}")
print(f"\nAnswer:\n{data2['answer'][:500]}...")

In [ ]:
# Step 3 — Full conversation (3 turns)
print("STEP 3 — FULL CONVERSATION")
print("=" * 40)

questions = [
    "What are the main components of a RAG system?",
    "Which component has the biggest impact on answer quality?",
    "How would you improve that component?",
]

conv_session = None

for i, question in enumerate(questions, 1):
    payload = {
        "query"     : question,
        "top_k"     : 3,
        "use_hybrid": True,
        "model"     : "llama3.2:1b",
    }
    if conv_session:
        payload["session_id"] = conv_session

    r = requests.post(f"{API}/ask", json=payload, timeout=120)
    d = r.json()
    conv_session = d["session_id"]

    print(f"[Turn {i}] Q: {question}")
    print(f"         A: {d['answer'][:200]}...")
    print()

print(f"Session ID used throughout: {conv_session}")

In [ ]:
# Step 4 — Streaming with session
print("STEP 4 — STREAMING WITH SESSION")
print("=" * 40)
print(f"Continuing conversation (session: {conv_session[:8]}...)\n")

stream_resp = requests.post(f"{API}/stream", json={
    "query"     : "Summarise what we have discussed so far.",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    "session_id": conv_session,
}, stream=True, timeout=120)

print("Streaming answer:")
print("-" * 40)

full_answer = ""
returned_session = None

for line in stream_resp.iter_lines():
    if not line:
        continue
    text = line.decode("utf-8")
    if not text.startswith("data: "):
        continue
    event = json.loads(text[6:])

    if "chunk" in event:
        print(event["chunk"], end="", flush=True)
        full_answer += event["chunk"]

    if event.get("done"):
        returned_session = event.get("session_id")
        break

print(f"\n" + "-" * 40)
print(f"session_id consistent: {returned_session == conv_session}")

In [ ]:
# Step 5 — Inspect Redis session history
print("STEP 5 — SESSION HISTORY IN REDIS")
print("=" * 40)

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
key = f"session:{conv_session}:history"
ttl = r.ttl(key)
raw = r.get(key)

if raw:
    history = json.loads(raw)
    print(f"Key     : {key}")
    print(f"TTL     : {ttl}s (~{ttl // 3600}h {(ttl % 3600) // 60}m remaining)")
    print(f"Messages: {len(history)} ({len(history) // 2} turns)")
    print()
    for msg in history:
        role = "You  " if msg["role"] == "user" else "Model"
        print(f"  [{role}] {msg['content'][:120]}")
else:
    print("No history found — check session_id or Redis connection")

In [ ]:
# Step 6 — New session (fresh context)
print("STEP 6 — NEW SESSION (FRESH CONTEXT)")
print("=" * 40)
print("Sending the same follow-up WITHOUT session_id.")
print("The model has no prior context and will ask for clarification.\n")

fresh_resp = requests.post(f"{API}/ask", json={
    "query"     : "Summarise what we have discussed so far.",
    "top_k"     : 3,
    "use_hybrid": True,
    "model"     : "llama3.2:1b",
    # no session_id
}, timeout=120)

fresh_data = fresh_resp.json()
new_session = fresh_data["session_id"]

print(f"Old session : {conv_session}")
print(f"New session : {new_session}")
print(f"Same session: {new_session == conv_session}")
print(f"\nAnswer (no context):\n{fresh_data['answer'][:400]}...")

In [ ]:
# Summary
print("MODULE 7 SUMMARY")
print("=" * 40)

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
session_keys = r.keys("session:*:history")
cache_keys   = r.keys("exact_cache:*")

print(f"Active sessions in Redis : {len(session_keys)}")
print(f"Exact-match cache entries: {len(cache_keys)}")
print()
print("What we covered:")
print("  ✓ Stateless request returns a new session_id")
print("  ✓ Passing session_id gives the model conversation context")
print("  ✓ /stream supports session_id identically to /ask")
print("  ✓ History is stored in Redis with a 24-hour TTL")
print("  ✓ A new session starts with no prior context")
print("  ✓ Exact-match cache is bypassed for session requests")